# Loading Data

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

#load datasets
wait_times = pd.read_csv('daily_wait_times.csv')
infections = pd.read_csv('infections.csv')

# Data Cleaning and Transformation

## wait_times columns

In [2]:
wait_times.dtypes

Hospital Name                  str
LHD                            str
Date                           str
Postcode                     int64
Average Persons Waiting    float64
dtype: object

### Changing 'Date' column
Based on the data types, hospital name, LHD, Postcode and Average Persons waiting are correct. However, Date should be changed to date

In [3]:
wait_times['Date'] = pd.to_datetime(
wait_times['Date'], format='%d/%m/%Y', errors='coerce')

## infections columns

In [4]:
infections.dtypes

LHD                  str
Postcode           int64
COVID Cases          str
RSV Cases            str
Influenza Cases    int64
Date                 str
dtype: object

### COVID and RSV Cases
Based on visual data inpections, the '<5' str is causing issues for the data type. Assuming COVID cases always exists within the community, <5 will be transformed to 4

In [5]:
infections['COVID Cases'] = pd.to_numeric(
    infections['COVID Cases'], errors='coerce').astype('Int64')

infections['RSV Cases'] = pd.to_numeric(
    infections['RSV Cases'], errors='coerce').astype('Int64')

In [6]:
infections['COVID Cases'] = infections['COVID Cases'].fillna(4).astype('Int64')
infections['RSV Cases'] = infections['RSV Cases'].fillna(4).astype('Int64')

### Date to date/time

In [7]:
infections['Date'] = pd.to_datetime(
    infections['Date'],
    format='%d/%m/%Y', errors='coerce'
)

## Reinspection of the two dataframes

In [8]:
wait_times.dtypes

Hospital Name                         str
LHD                                   str
Date                       datetime64[us]
Postcode                            int64
Average Persons Waiting           float64
dtype: object

In [9]:
infections.dtypes

LHD                           str
Postcode                    int64
COVID Cases                 Int64
RSV Cases                   Int64
Influenza Cases             int64
Date               datetime64[us]
dtype: object

# Joining the two dataframes
Upon visual inspection, the two dataframes cannot be joined based on date and hospital.

## Date Cleanup
Two options:
1. Weekly Reporting: The easiest option, grouping by week
2. Daily Reporting: Operationally, the most operationally accurate reporting, EDs need to be able to allocate resources on a daily basis

In [10]:
#Given date x, generate dates from x-6 to x (periods = 7)
infections['Date'] = infections['Date'].apply(
    lambda x: pd.date_range(end=x, periods=7)
)

In [11]:
#Explodes the date range into individual rows for each date
infections = infections.explode('Date').reset_index(drop=True)

## LHD Columns

In [12]:
#compare the LHD columns in both dataframes
wait_times['LHD'].equals(infections['LHD'])

False

In [13]:
wait_times_LHD = set(wait_times['LHD'])
infections_LHD = set(infections['LHD'])

print("Present in wait_times but not infections: " + str(wait_times_LHD - infections_LHD))

Present in wait_times but not infections: {"St Vincent's Health Network", "Sydney Children's Hospitals Network"}


In [14]:
print("Present in infections but not wait_times: " + str(infections_LHD - wait_times_LHD))

Present in infections but not wait_times: set()


### Identifying hospitals in SCHN and St. Vincents Network
Based on the comparison, the wait_times df has 2 extra LHDs. This section identifies the hospitals within that LHD and re-assigns them based on geographic location

In [15]:
print(wait_times.loc[
        wait_times['LHD'] == "Sydney Children's Hospitals Network",
        'Hospital Name'
    ].unique())

<ArrowStringArray>
['Sydney Children's Hospital', 'The Children's Hospital at Westmead']
Length: 2, dtype: str


In [16]:
print(wait_times.loc[
        wait_times['LHD'] == "St Vincent's Health Network",
        'Hospital Name'
    ].unique())

<ArrowStringArray>
['St Vincent's Hospital']
Length: 1, dtype: str


### Re-Assigning LHD based on geographic location
Based on the following mappings: \
SCH -> South Eastern Sydney. \
TCH Westmead -> Western Sydney. \
St. Vincents -> Sydney. 

In [17]:
wait_times.loc[
    wait_times['Hospital Name'] == "Sydney Children's Hospital",
    'LHD'
] = 'South Eastern Sydney'

In [18]:
wait_times.loc[
    wait_times['Hospital Name'] == "The Children's Hospital at Westmead",
    'LHD'
] = 'Western Sydney'

In [19]:
wait_times.loc[
    wait_times['Hospital Name'] == "St Vincent's Hospital",
    'LHD'
] = 'Sydney'

### Re-examining LHD

In [20]:
wait_times_LHD = set(wait_times['LHD'])
infections_LHD = set(infections['LHD'])
print("Present in wait_times but not infections: " + str(wait_times_LHD - infections_LHD))

Present in wait_times but not infections: set()


## Joining Data on date and LHD

In [21]:
infection_wait_times = pd.merge(left=wait_times, right=infections, how='left', on=['Date', 'LHD'])

In [22]:
#drop rows where Hospital Name is missing
infection_wait_times.dropna(subset=['Hospital Name'], inplace=True)

# Cleaning the joined dataframe

In [23]:
infection_wait_times.dtypes

Hospital Name                         str
LHD                                   str
Date                       datetime64[us]
Postcode_x                          int64
Average Persons Waiting           float64
Postcode_y                        float64
COVID Cases                         Int64
RSV Cases                           Int64
Influenza Cases                   float64
dtype: object

In [26]:
#drop postcode_y as it is the postcode for the LHD
infection_wait_times.drop(columns=['Postcode_y'], inplace=True)


In [24]:
#change the data type of 'Influenza Cases' 
infection_wait_times['Influenza Cases'] = infection_wait_times['Influenza Cases'].astype('Int64')

In [27]:
infection_wait_times.isna().sum()

Hospital Name                0
LHD                          0
Date                         0
Postcode_x                   0
Average Persons Waiting      0
COVID Cases                472
RSV Cases                  472
Influenza Cases            472
dtype: int64

In [31]:
#for rows without covid cases, use median imputation
#obtain median value
median_covid = infection_wait_times['COVID Cases'].median()
median_rsv = infection_wait_times['RSV Cases'].median()
median_influenza = infection_wait_times['Influenza Cases'].median()

print(median_covid, median_rsv, median_influenza)

43.0 68.0 180.0


In [32]:
infection_wait_times["COVID Cases"] = (
    infection_wait_times["COVID Cases"]
    .fillna(median_covid)
    .round()
    .astype("Int64")
)


infection_wait_times["RSV Cases"] = (
    infection_wait_times["RSV Cases"]
    .fillna(median_rsv)
    .round()
    .astype("Int64")
)

infection_wait_times["Influenza Cases"] = (
    infection_wait_times["Influenza Cases"]
    .fillna(median_influenza)
    .round()
    .astype("Int64")
)

In [33]:
infection_wait_times = infection_wait_times.rename(columns={'Postcode_x': 'Postcode'})

In [34]:
#reorder
order = [
    "Date",
    "LHD",
    "Hospital Name",
    "Postcode",
    "COVID Cases",
    "RSV Cases",
    "Influenza Cases",
    "Average Persons Waiting"
]

infection_wait_times = infection_wait_times[order]

In [35]:
infection_wait_times.dtypes

Date                       datetime64[us]
LHD                                   str
Hospital Name                         str
Postcode                            int64
COVID Cases                         Int64
RSV Cases                           Int64
Influenza Cases                     Int64
Average Persons Waiting           float64
dtype: object